# Checkpoint 2 - GenAI GANs
## Generative AI e Advanced Nets - José Maia

---

## Integrantes

| Nome | RM |
|---|---|
| Guilherme Gama | RM565293 |
| Bruno Fernandes Nascimento | RM552574 |
| Edgar Lódula de Assis | RM565260 |
| Júlia Aben-Athar | RM566325 |
| Igor Thiago Nakajima Vieira | RM563632 |


# Analisador de Contratos
**RAG + Saídas Estruturadas com GPT-4o-mini**

In [14]:
%pip install --quiet --upgrade \
    langchain langchain-community langchain-openai langchain-core \
    langchain-text-splitters langgraph pypdf python-dotenv

In [15]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("Chave carregada do Colab Secrets.")
except ImportError:
    # Local: carrega do .env
    from dotenv import load_dotenv
    load_dotenv()
    print("Chave carregada do .env.")

Chave carregada do Colab Secrets.


In [16]:
import json
import tempfile
from typing import List

from langchain.chat_models import init_chat_model
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import START, StateGraph
from typing_extensions import TypedDict

## Schemas e Queries



In [17]:
SCHEMA_CONTRATO = {
    "comprador": {
        "nome_completo": "",
        "cpf": "",
        "rg": "",
        "nacionalidade": "",
        "estado_civil": "",
        "conjuge": {"nome": "", "cpf": "", "rg": ""},
        "profissao": "",
        "endereco": "",
        "email": ""
    },
    "vendedor": {
        "nome_completo": "",
        "cnpj": "",
        "endereco_sede": "",
        "representantes_legais": []
    },
    "imovel": {
        "endereco_completo": "",
        "matricula": "",
        "cartorio_registro": "",
        "inscricao_iptu": "",
        "area_total_m2": "",
        "areas_lazer": "",
        "comodos": "",
        "vagas_garagem": ""
    },
    "condicoes_financeiras": {
        "preco_total": "",
        "indice_reajuste": "",
        "periodicidade_reajuste": "",
        "juros_mora": "",
        "multa_atraso": "",
        "comissao_corretagem": ""
    }
}

QUERIES = {
    "comprador": "nome completo CPF RG nacionalidade estado civil cônjuge profissão endereço email comprador",
    "vendedor": "vendedor incorporadora CNPJ endereço sede representantes legais",
    "imovel": "imóvel matrícula cartório registro IPTU área total lazer cômodos vagas garagem endereço",
    "condicoes_financeiras": "preço total valor venda índice reajuste INCC IGPM IPCA juros multa atraso comissão corretagem"
}

## Prompt


In [18]:
PROMPT = ChatPromptTemplate.from_messages([
    ("human", """Você é um especialista em análise de contratos jurídicos brasileiros.

Analise os trechos do contrato abaixo e extraia SOMENTE as informações do papel solicitado na questão.

REGRAS DE VALIDAÇÃO DE PAPEL:
- Se a questão pede dados do COMPRADOR: extraia apenas campos explicitamente atribuídos ao COMPRADOR no texto. Ignore qualquer dado do VENDEDOR/INCORPORADORA.
- Se a questão pede dados do VENDEDOR/INCORPORADORA: extraia apenas campos explicitamente atribuídos ao VENDEDOR ou INCORPORADORA. Ignore dados do COMPRADOR.
- Se a questão pede dados do IMÓVEL: extraia apenas características e identificação do imóvel. Ignore dados das partes.
- Se a questão pede condições financeiras: extraia apenas valores, índices e penalidades contratuais. Ignore dados pessoais.
- Extraia APENAS o que estiver explicitamente no texto. Não infira nem complete com suposições.
- Para campos não encontrados, use string vazia \"\" ou lista vazia [].
- Retorne SOMENTE um objeto JSON válido. Sem explicações fora do JSON.

Schema esperado:
{schema}

Question: {question}
Context: {context}
Answer:""")
])

## Funções do Pipeline RAG

In [19]:
def build_vector_store(pdf_path: str):
    embeddings = OpenAIEmbeddings(model="text-embedding-ada-002")
    vector_store = InMemoryVectorStore(embeddings)

    loader = PyPDFLoader(pdf_path)
    docs = loader.load()
    print(f"PDF carregado: {len(docs)} página(s)")

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        add_start_index=True,
    )
    all_splits = splitter.split_documents(docs)
    print(f"Chunks gerados: {len(all_splits)}")

    vector_store.add_documents(all_splits)
    print("Vector store indexado.")
    return vector_store, all_splits


def _search_terms(value) -> list:
    if isinstance(value, dict):
        texts = [str(v) for v in value.values() if v and str(v).strip()]
    elif isinstance(value, list):
        texts = [str(v) for v in value if v and str(v).strip()]
    else:
        texts = [str(value)] if value else []

    tokens = set()
    for text in texts:
        for token in text.split():
            if len(token) > 3 or any(c.isdigit() for c in token):
                tokens.add(token.lower())
    return list(tokens)


def find_source_chunks(value, all_splits: list, k: int = 2) -> list:
    terms = _search_terms(value)
    if not terms:
        return []

    scored = []
    for split in all_splits:
        content_lower = split.page_content.lower()
        hits = sum(1 for t in terms if t in content_lower)
        if hits > 0:
            scored.append((hits, split))

    if not scored:
        return []

    scored.sort(key=lambda x: x[0], reverse=True)
    max_hits = scored[0][0]
    threshold = max(1, max_hits - 1)
    return [s for hits, s in scored[:k] if hits >= threshold]

## Pipeline Principal



In [20]:
def build_rag_graph(llm, vector_store):
    class State(TypedDict):
        question: str
        schema: str
        context: List[Document]
        answer: str

    def retrieve(state: State):
        return {"context": vector_store.similarity_search(state["question"], k=4)}

    def generate(state: State):
        docs_content = ""
        for i, doc in enumerate(state["context"], 1):
            pos = doc.metadata.get("start_index", "?")
            docs_content += f"[Trecho {i} — posição {pos} no documento]\n{doc.page_content}\n\n"

        messages = PROMPT.invoke({
            "question": state["question"],
            "context":  docs_content,
            "schema":   state["schema"]
        })
        return {"answer": llm.invoke(messages).content}

    graph = StateGraph(State).add_sequence([retrieve, generate])
    graph.add_edge(START, "retrieve")
    return graph.compile()

## Upload e Execução



In [21]:
try:
    from google.colab import files
    uploaded = files.upload()
    pdf_path = list(uploaded.keys())[0]
    print(f"Arquivo: {pdf_path}")
except ImportError:
    pdf_path = "contrato.pdf"
    print(f"Usando arquivo local: {pdf_path}")

Saving contrato.pdf to contrato (2).pdf
Arquivo: contrato (2).pdf


In [22]:
vector_store, all_splits = build_vector_store(pdf_path)

PDF carregado: 10 página(s)
Chunks gerados: 21
Vector store indexado.


In [23]:
llm = init_chat_model("gpt-4o-mini", model_provider="openai")
graph = build_rag_graph(llm, vector_store)

resultado_final = {}

for secao, query in QUERIES.items():
    print(f"\nAnalisando: {secao}...")

    result = graph.invoke({
        "question": f"Extraia as informações de '{secao}' do contrato: {query}",
        "schema":   json.dumps(SCHEMA_CONTRATO[secao], ensure_ascii=False, indent=2)
    })

    raw = result["answer"].replace("```json", "").replace("```", "").strip()
    try:
        dados = json.loads(raw)
    except json.JSONDecodeError:
        dados = raw

    trechos_por_campo = {}
    if isinstance(dados, dict):
        for field, field_value in dados.items():
            source_docs = find_source_chunks(field_value, all_splits, k=2)
            trechos_por_campo[field] = [
                {
                    "trecho":  doc.page_content,
                    "posicao": doc.metadata.get("start_index", "?"),
                    "pagina":  doc.metadata.get("page", "?")
                }
                for doc in source_docs
            ]

    resultado_final[secao] = {
        "dados":            dados,
        "trechos_por_campo": trechos_por_campo
    }

print("\nAnálise concluída!")


Analisando: comprador...

Analisando: vendedor...

Analisando: imovel...

Analisando: condicoes_financeiras...

Análise concluída!


## Resultados

In [26]:
from IPython.display import display, HTML
import html as html_lib

FIELD_LABELS = {
    "nome_completo":         "Nome Completo",
    "cpf":                   "CPF",
    "rg":                    "RG",
    "nacionalidade":         "Nacionalidade",
    "estado_civil":          "Estado Civil",
    "conjuge":               "Cônjuge",
    "profissao":             "Profissão",
    "endereco":              "Endereço",
    "email":                 "E-mail",
    "cnpj":                  "CNPJ",
    "endereco_sede":         "Endereço da Sede",
    "representantes_legais": "Representantes Legais",
    "endereco_completo":     "Endereço Completo",
    "matricula":             "Matrícula",
    "cartorio_registro":     "Cartório de Registro",
    "inscricao_iptu":        "Inscrição IPTU",
    "area_total_m2":         "Área Total (m²)",
    "areas_lazer":           "Áreas de Lazer",
    "comodos":               "Cômodos",
    "vagas_garagem":         "Vagas de Garagem",
    "preco_total":           "Preço Total",
    "indice_reajuste":       "Índice de Reajuste",
    "periodicidade_reajuste":"Periodicidade do Reajuste",
    "juros_mora":            "Juros de Mora",
    "multa_atraso":          "Multa por Atraso",
    "comissao_corretagem":   "Comissão de Corretagem"
}

SECTION_LABELS = {
    "comprador":             "Comprador",
    "vendedor":              "Vendedor / Incorporadora",
    "imovel":                "Imóvel",
    "condicoes_financeiras": "Condições Financeiras"
}

def render_value(v) -> str:
    if isinstance(v, dict):
        parts = [f"{k}: {val}" for k, val in v.items() if val]
        return ", ".join(parts) if parts else ""
    if isinstance(v, list):
        return "; ".join(str(x) for x in v) if v else ""
    return str(v) if v else ""

def highlight_value(value: str, text: str, context: int = 200) -> str:
    if not value or not text:
        return html_lib.escape(text[:400] if text else "")
    text_lower = text.lower()
    term = value.strip()
    pos = text_lower.find(term.lower())
    if pos == -1:
        tokens = sorted(
            [t for t in term.split() if len(t) > 4 or any(c.isdigit() for c in t)],
            key=len, reverse=True
        )
        for token in tokens:
            p = text_lower.find(token.lower())
            if p != -1:
                pos, term = p, token
                break
    if pos == -1:
        return html_lib.escape(text[:400])
    ws = max(0, pos - context)
    we = min(len(text), pos + len(term) + context)
    window = text[ws:we]
    mp = window.lower().find(term.lower())
    if mp != -1:
        return (
            ("..." if ws > 0 else "") +
            html_lib.escape(window[:mp]) +
            f'<mark style="background:#fef08a;padding:1px 4px;border-radius:3px;font-weight:700">' +
            html_lib.escape(window[mp:mp+len(term)]) +
            "</mark>" +
            html_lib.escape(window[mp+len(term):]) +
            ("..." if we < len(text) else "")
        )
    return html_lib.escape(window)

def render_resultado(resultado: dict):
    for secao, conteudo in resultado.items():
        dados = conteudo["dados"]
        trechos_por_campo = conteudo["trechos_por_campo"]
        titulo = SECTION_LABELS.get(secao, secao)

        html = f"""
        <h2 style="border-bottom:2px solid #3b82f6;padding-bottom:6px;color:#93c5fd">{titulo}</h2>
        <table style="width:100%;border-collapse:collapse;font-family:sans-serif;font-size:14px;color:#e2e8f0">
          <thead>
            <tr style="background:#1e293b">
              <th style="text-align:left;padding:8px 12px;width:30%;color:#e2e8f0">Campo</th>
              <th style="text-align:left;padding:8px 12px;color:#e2e8f0">Valor</th>
            </tr>
          </thead>
          <tbody>
        """

        if isinstance(dados, dict):
            for field, value in dados.items():
                label = FIELD_LABELS.get(field, field)
                rendered = render_value(value)
                valor_html = html_lib.escape(rendered) if rendered else "<em style='color:#64748b'>Não encontrado</em>"

                trechos = trechos_por_campo.get(field, [])
                trecho_html = ""
                if rendered and trechos:
                    trecho_html = "<br>"
                    for i, t in enumerate(trechos, 1):
                        hl = highlight_value(rendered, t["trecho"])
                        trecho_html += f"""
                        <details style="margin-top:6px">
                          <summary style="cursor:pointer;color:#60a5fa;font-size:12px">
                            Trecho {i} — Página {t['pagina']} · posição {t['posicao']}
                          </summary>
                          <pre style="background:#0f172a;border:1px solid #334155;border-radius:4px;
                                      padding:10px;font-size:12px;white-space:pre-wrap;
                                      line-height:1.6;margin-top:4px;color:#e2e8f0">{hl}</pre>
                        </details>
                        """

                html += f"""
                <tr style="border-bottom:1px solid #334155">
                  <td style="padding:10px 12px;font-weight:500;color:#94a3b8;vertical-align:top">{label}</td>
                  <td style="padding:10px 12px;vertical-align:top;color:#e2e8f0">{valor_html}{trecho_html}</td>
                </tr>
                """

        html += "</tbody></table><br>"
        display(HTML(html))

render_resultado(resultado_final)

Campo,Valor
Nome Completo,"CARLOS HENRIQUE SOUZA Trecho 1 — Página 0 · posição 0 ... RG nº 23.456.789-1 SSP/SP, inscrita no CPF/MF sob nº 234.567.890-11, residente e domiciliada no mesmo endereço acima; doravante denominados simplesmente VENDEDORES; e, de outro lado: COMPRADOR: CARLOS HENRIQUE SOUZA, brasileiro, solteiro, analista de sistemas, portador do RG nº 34.567.890-2 SSP/SP, inscrito no CPF/MF sob nº 345.678.901-22, residente e domiciliado à Avenida Brasil, nº 980, apartamento 31, Bairr... Trecho 2 — Página 9 · posição 0 VENDEDORES JOÃO CARLOS ALMEIDA CPF nº 123.456.789-00 MARIA FERNANDA ALMEIDA CPF nº 234.567.890-11 COMPRADOR CARLOS HENRIQUE SOUZA CPF nº 345.678.901-22 TESTEMUNHAS 1. Nome: Renata Lima Martins CPF: 456.789.012-33 RG: 45.678.901-3 SSP/SP 2. Nome: Paulo Roberto Nogueira CPF: 567.890.123-44 RG: 56.789.012-4 SSP/SP"
CPF,"345.678.901-22 Trecho 1 — Página 0 · posição 0 ...enominados simplesmente VENDEDORES; e, de outro lado: COMPRADOR: CARLOS HENRIQUE SOUZA, brasileiro, solteiro, analista de sistemas, portador do RG nº 34.567.890-2 SSP/SP, inscrito no CPF/MF sob nº 345.678.901-22, residente e domiciliado à Avenida Brasil, nº 980, apartamento 31, Bairro Centro, Campinas/SP, CEP 13010-001; Trecho 2 — Página 0 · posição 795 portador do RG nº 34.567.890-2 SSP/SP, inscrito no CPF/MF sob nº 345.678.901-22, residente e domiciliado à Avenida Brasil, nº 980, apartamento 31, Bairro Centro, Campinas/SP, CEP 13010-001; doravante denominado simplesmente COMPRADOR; têm entre si justo e contratado o presen..."
RG,"34.567.890-2 Trecho 1 — Página 0 · posição 0 ...iciliada no mesmo endereço acima; doravante denominados simplesmente VENDEDORES; e, de outro lado: COMPRADOR: CARLOS HENRIQUE SOUZA, brasileiro, solteiro, analista de sistemas, portador do RG nº 34.567.890-2 SSP/SP, inscrito no CPF/MF sob nº 345.678.901-22, residente e domiciliado à Avenida Brasil, nº 980, apartamento 31, Bairro Centro, Campinas/SP, CEP 13010-001; Trecho 2 — Página 0 · posição 795 portador do RG nº 34.567.890-2 SSP/SP, inscrito no CPF/MF sob nº 345.678.901-22, residente e domiciliado à Avenida Brasil, nº 980, apartamento 31, Bairro Centro, Campinas/SP, CEP 13010-001; doravante denominado simplesmente COM..."
Nacionalidade,"brasileiro Trecho 1 — Página 0 · posição 0 CONTRATO PARTICULAR DE COMPROMISSO DE COMPRA E VENDA DE IMÓVEL Pelo presente instrumento particular, de um lado: VENDEDOR: JOÃO CARLOS ALMEIDA, brasileiro, casado sob o regime de comunhão parcial de bens, engenheiro civil, portador do RG nº 12.345.678-9 SSP/SP, inscrito no CPF/MF sob nº 123.456.789-00, residente e domiciliado à Rua das Palmeiras, nº ..."
Estado Civil,"solteiro Trecho 1 — Página 0 · posição 0 ...ta no CPF/MF sob nº 234.567.890-11, residente e domiciliada no mesmo endereço acima; doravante denominados simplesmente VENDEDORES; e, de outro lado: COMPRADOR: CARLOS HENRIQUE SOUZA, brasileiro, solteiro, analista de sistemas, portador do RG nº 34.567.890-2 SSP/SP, inscrito no CPF/MF sob nº 345.678.901-22, residente e domiciliado à Avenida Brasil, nº 980, apartamento 31, Bairro Centro, Campinas/SP..."
Cônjuge,Não encontrado
Profissão,"analista de sistemas Trecho 1 — Página 0 · posição 0 ...MF sob nº 234.567.890-11, residente e domiciliada no mesmo endereço acima; doravante denominados simplesmente VENDEDORES; e, de outro lado: COMPRADOR: CARLOS HENRIQUE SOUZA, brasileiro, solteiro, analista de sistemas, portador do RG nº 34.567.890-2 SSP/SP, inscrito no CPF/MF sob nº 345.678.901-22, residente e domiciliado à Avenida Brasil, nº 980, apartamento 31, Bairro Centro, Campinas/SP, CEP 13010-001;"
Endereço,"Avenida Brasil, nº 980, apartamento 31, Bairro Centro, Campinas/SP, CEP 13010-001 Trecho 1 — Página 0 · posição 0 ...o, solteiro, analista de sistemas, portador do RG nº 34.567.890-2 SSP/SP, inscrito no CPF/MF sob nº 345.678.901-22, residente e domiciliado à Avenida Brasil, nº 980, apartamento 31, Bairro Centro, Campinas/SP, CEP 13010-001; Trecho 2 — Página 0 · posição 795 portador do 

Campo,Valor
Nome Completo,"JOÃO CARLOS ALMEIDA Trecho 1 — Página 0 · posição 0 CONTRATO PARTICULAR DE COMPROMISSO DE COMPRA E VENDA DE IMÓVEL Pelo presente instrumento particular, de um lado: VENDEDOR: JOÃO CARLOS ALMEIDA, brasileiro, casado sob o regime de comunhão parcial de bens, engenheiro civil, portador do RG nº 12.345.678-9 SSP/SP, inscrito no CPF/MF sob nº 123.456.789-00, residente e domiciliado à Rua das Pa... Trecho 2 — Página 9 · posição 0 VENDEDORES JOÃO CARLOS ALMEIDA CPF nº 123.456.789-00 MARIA FERNANDA ALMEIDA CPF nº 234.567.890-11 COMPRADOR CARLOS HENRIQUE SOUZA CPF nº 345.678.901-22 TESTEMUNHAS 1. Nome: Renata Lima Martins CPF: 456.789.012-3..."
CNPJ,Não encontrado
Endereço da Sede,"Rua das Palmeiras, nº 150, apartamento 82, Bairro Jardim Europa, São Paulo/SP, CEP 01449-000 Trecho 1 — Página 0 · posição 0 ...do sob o regime de comunhão parcial de bens, engenheiro civil, portador do RG nº 12.345.678-9 SSP/SP, inscrito no CPF/MF sob nº 123.456.789-00, residente e domiciliado à Rua das Palmeiras, nº 150, apartamento 82, Bairro Jardim Europa, São Paulo/SP, CEP 01449-000; e sua esposa/anuente: MARIA FERNANDA ALMEIDA, brasileira, casada, administradora, portadora do RG nº 23.456.789-1 SSP/SP, inscrita no CPF/MF ..."
Representantes Legais,Não encontrado


Campo,Valor
Endereço Completo,"Rua das Acácias, nº 245, Casa 03, Condomínio Residencial Vila Serena, Bairro Jardim Primavera, Campinas/SP, CEP 13092-000 Trecho 1 — Página 0 · posição 795 portador do RG nº 34.567.890-2 SSP/SP, inscrito no CPF/MF sob nº 345.678.901-22, residente e domiciliado à Avenida Brasil, nº 980, apartamento 31, Bairro Centro, Campinas/SP, CEP 13010-001; doravante denominado simplesmente COMPRADOR; têm entre si justo e contratado o presente Contrato Particular de Compromisso de Compra e Venda de Imóvel, mediante as cláusulas e condi..."
Matrícula,"89.765 Trecho 1 — Página 0 · posição 795 ... de 220,00 m², contendo 3 dormitórios, sendo 1 suíte, sala de estar, sala de jantar, cozinha, lavanderia, quintal, 2 banheiros sociais e 2 vagas de garagem. O imóvel encontra-se matriculado sob nº 89.765 no 2º Cartório de Registro de Imóveis da Trecho 2 — Página 0 · posição 1622 lavanderia, quintal, 2 banheiros sociais e 2 vagas de garagem. O imóvel encontra-se matriculado sob nº 89.765 no 2º Cartório de Registro de Imóveis da Comarca de Campinas/SP, com inscrição municipal nº 2445.7789.0012.0000."
Cartório de Registro,"2º Cartório de Registro de Imóveis da Comarca de Campinas/SP Trecho 1 — Página 0 · posição 1622 lavanderia, quintal, 2 banheiros sociais e 2 vagas de garagem. O imóvel encontra-se matriculado sob nº 89.765 no 2º Cartório de Registro de Imóveis da Comarca de Campinas/SP, com inscrição municipal nº 2445.7789.0012.0000. Trecho 2 — Página 0 · posição 795 portador do RG nº 34.567.890-2 SSP/SP, inscrito no CPF/MF sob nº 345.678.901-22, residente e domiciliado à Avenida Brasil, nº 980, apartamento 31, Bairro Centro, Campinas/SP, CEP 13010-001; doravante denominado simplesmente COMPRADOR; têm entre si justo e contratado o presente Contrato Particular de Compromisso de Compra e Venda de Imóvel, mediante as cláusulas e cond..."
Inscrição IPTU,"2445.7789.0012.0000 Trecho 1 — Página 0 · posição 1622 ...nderia, quintal, 2 banheiros sociais e 2 vagas de garagem. O imóvel encontra-se matriculado sob nº 89.765 no 2º Cartório de Registro de Imóveis da Comarca de Campinas/SP, com inscrição municipal nº 2445.7789.0012.0000."
Área Total (m²),"220,00 Trecho 1 — Página 0 · posição 795 ...º 245, Casa 03, Condomínio Residencial Vila Serena, Bairro Jardim Primavera, Campinas/SP, CEP 13092-000. O imóvel possui área privativa construída aproximada de 145,00 m², área total de terreno de 220,00 m², contendo 3 dormitórios, sendo 1 suíte, sala de estar, sala de jantar, cozinha, lavanderia, quintal, 2 banheiros sociais e 2 vagas de garagem. O imóvel encontra-se matriculado sob nº 89.765 no 2..."
Áreas de Lazer,Não encontrado
Cômodos,"3 dormitórios, sendo 1 suíte, sala de estar, sala de jantar, cozinha, lavanderia, quintal, 2 banheiros sociais Trecho 1 — Página 0 · posição 795 ...ínio Residencial Vila Serena, Bairro Jardim Primavera, Campinas/SP, CEP 13092-000. O imóvel possui área privativa construída aproximada de 145,00 m², área total de terreno de 220,00 m², contendo 3 dormitórios, sendo 1 suíte, sala de estar, sala de jantar, cozinha, lavanderia, quintal, 2 banheiros sociais e 2 vagas de garagem. O imóvel encontra-se matriculado sob nº 89.765 no 2º Cartório de Registro de Im..."
Vagas de Garagem,"2 Trecho 1 — Página 0 · posição 0 ... VENDA DE IMÓVEL Pelo presente instrumento particular, de um lado: VENDEDOR: JOÃO CARLOS ALMEIDA, brasileiro, casado sob o regime de comunhão parcial de bens, engenheiro civil, portador do RG nº 12.345.678-9 SSP/SP, inscrito no CPF/MF sob nº 123.456.789-00, residente e domiciliado à Rua das Palmeiras, nº 150, apartamento 82, Bairro Jardim Europa, São Paulo/SP, CEP 01449-000; e sua esposa/anu... Trecho 2 — Página 0 · posição 795 portador do RG nº 34.567.890-2 SSP/SP, inscrito no CPF/MF sob nº 345.678.901-22, residente e domiciliado à Avenida Brasil, nº 980, apartamento 31, Bairro Centro, Campinas/SP, CEP 13010-001; doravante denominado simplesmente COM..."


Campo,Valor
Preço Total,"R$ 850.000,00 Trecho 1 — Página 1 · posição 0 ...que identificada posteriormente, os VENDEDORES serão responsáveis por sua integral regularização e pagamento. CLÁUSULA 3ª — DO PREÇO O preço certo e ajustado para a compra e venda do imóvel é de R$ 850.000,00 — oitocentos e cinquenta mil reais. O COMPRADOR pagará o preço aos VENDEDORES da seguinte forma: Trecho 2 — Página 1 · posição 740 CLÁUSULA 3ª — DO PREÇO O preço certo e ajustado para a compra e venda do imóvel é de R$ 850.000,00 — oitocentos e cinquenta mil reais. O COMPRADOR pagará o preço aos VENDEDORES da seguinte forma: a) R$ 85.000,00 — oitenta e cinco mil reais — a título de sinal e princípio de pagamento, pagos no..."
Índice de Reajuste,"IPCA/IBGE Trecho 1 — Página 5 · posição 0 ...uer parcela pelo COMPRADOR acarretará incidência de multa moratória de 2% sobre o valor em atraso, acrescida de juros de mora de 1% ao mês, calculados pro rata die, e correção monetária pelo índice IPCA/IBGE, ou outro índice que venha a substituí-lo. Caso o atraso ultrapasse 30 dias corridos, os VENDEDORES poderão considerar rescindido o presente contrato, mediante notificação por escrito ao COMPRADOR... Trecho 2 — Página 5 · posição 772 ...cumprimento de prazo essencial previsto neste contrato. Em caso de rescisão por culpa dos VENDEDORES, estes deverão restituir ao COMPRADOR todos os valores recebidos, corrigidos monetariamente pelo IPCA/IBGE, sem prejuízo da multa contratual prevista neste instrumento. Em caso de rescisão por culpa do COMPRADOR, os VENDEDORES poderão reter o sinal pago e cobrar eventuais prejuízos adicionais devidamen..."
Periodicidade do Reajuste,Não encontrado
Juros de Mora,"1% ao mês Trecho 1 — Página 5 · posição 0 ...ES. CLÁUSULA 13ª — DA INADIMPLÊNCIA O atraso no pagamento de qualquer parcela pelo COMPRADOR acarretará incidência de multa moratória de 2% sobre o valor em atraso, acrescida de juros de mora de 1% ao mês, calculados pro rata die, e correção monetária pelo índice IPCA/IBGE, ou outro índice que venha a substituí-lo. Caso o atraso ultrapasse 30 dias corridos, os VENDEDORES poderão considerar rescindi..."
Multa por Atraso,"2% Trecho 1 — Página 5 · posição 0 ...r dolo, má-fé ou omissão relevante por parte dos VENDEDORES. CLÁUSULA 13ª — DA INADIMPLÊNCIA O atraso no pagamento de qualquer parcela pelo COMPRADOR acarretará incidência de multa moratória de 2% sobre o valor em atraso, acrescida de juros de mora de 1% ao mês, calculados pro rata die, e correção monetária pelo índice IPCA/IBGE, ou outro índice que venha a substituí-lo. Caso o atraso ultra..."
Comissão de Corretagem,"R$ 42.500,00 Trecho 1 — Página 6 · posição 0 ...laram que a presente negociação foi intermediada pela corretora IMOBILIÁRIA HORIZONTE LTDA., inscrita no CNPJ sob nº 12.345.678/0001-90, CRECI/SP nº 45.678-J. A comissão de corretagem, no valor de R$ 42.500,00 — quarenta e dois mil e quinhentos reais — correspondente a 5% do valor da venda, será paga pelos VENDEDORES diretamente à imobiliária, na data do recebimento do sinal."


## 10. Resultado bruto (JSON)

In [25]:
for secao, conteudo in resultado_final.items():
    print(f"\n{'='*60}")
    print(f"  {SECTION_LABELS.get(secao, secao).upper()}")
    print(f"{'='*60}")
    print(json.dumps(conteudo["dados"], ensure_ascii=False, indent=2))


  COMPRADOR
{
  "nome_completo": "CARLOS HENRIQUE SOUZA",
  "cpf": "345.678.901-22",
  "rg": "34.567.890-2",
  "nacionalidade": "brasileiro",
  "estado_civil": "solteiro",
  "conjuge": {
    "nome": "",
    "cpf": "",
    "rg": ""
  },
  "profissao": "analista de sistemas",
  "endereco": "Avenida Brasil, nº 980, apartamento 31, Bairro Centro, Campinas/SP, CEP 13010-001",
  "email": "carlos.souza@emailficticio.com"
}

  VENDEDOR / INCORPORADORA
{
  "nome_completo": "JOÃO CARLOS ALMEIDA",
  "cnpj": "",
  "endereco_sede": "Rua das Palmeiras, nº 150, apartamento 82, Bairro Jardim Europa, São Paulo/SP, CEP 01449-000",
  "representantes_legais": []
}

  IMÓVEL
{
  "endereco_completo": "Rua das Acácias, nº 245, Casa 03, Condomínio Residencial Vila Serena, Bairro Jardim Primavera, Campinas/SP, CEP 13092-000",
  "matricula": "89.765",
  "cartorio_registro": "2º Cartório de Registro de Imóveis da Comarca de Campinas/SP",
  "inscricao_iptu": "2445.7789.0012.0000",
  "area_total_m2": "220,00",
  